In [18]:
# ==========================================
# HOUSE PRICE PREDICTION PROJECT
# Complete Code with Dataset Included
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================
# CREATE DATASET IN CODE
# ==========================================

np.random.seed(42)

n_train = 1000
n_test = 300

# Train Dataset
train = pd.DataFrame({
    "Id": range(1, n_train + 1),
    "LotArea": np.random.randint(1000, 15000, n_train),
    "OverallQual": np.random.randint(1, 10, n_train),
    "OverallCond": np.random.randint(1, 10, n_train),
    "YearBuilt": np.random.randint(1970, 2024, n_train),
    "BedroomAbvGr": np.random.randint(1, 6, n_train),
    "FullBath": np.random.randint(1, 4, n_train),
    "GarageCars": np.random.randint(0, 4, n_train),
    "GarageArea": np.random.randint(100, 1000, n_train),
    "GrLivArea": np.random.randint(500, 4000, n_train),
    "TotalBsmtSF": np.random.randint(300, 2500, n_train),
    "FirstFlrSF": np.random.randint(500, 2500, n_train),
    "TotRmsAbvGrd": np.random.randint(2, 12, n_train)
})

# Target Variable
train["SalePrice"] = (
        50000
        + train["LotArea"] * 8
        + train["OverallQual"] * 25000
        + train["GarageCars"] * 15000
        + train["GrLivArea"] * 40
        + train["GarageArea"] * 20
        + np.random.randint(-50000, 50000, n_train)
)

# Test Dataset
test = pd.DataFrame({
    "Id": range(n_train + 1, n_train + n_test + 1),
    "LotArea": np.random.randint(1000, 15000, n_test),
    "OverallQual": np.random.randint(1, 10, n_test),
    "OverallCond": np.random.randint(1, 10, n_test),
    "YearBuilt": np.random.randint(1970, 2024, n_test),
    "BedroomAbvGr": np.random.randint(1, 6, n_test),
    "FullBath": np.random.randint(1, 4, n_test),
    "GarageCars": np.random.randint(0, 4, n_test),
    "GarageArea": np.random.randint(100, 1000, n_test),
    "GrLivArea": np.random.randint(500, 4000, n_test),
    "TotalBsmtSF": np.random.randint(300, 2500, n_test),
    "FirstFlrSF": np.random.randint(500, 2500, n_test),
    "TotRmsAbvGrd": np.random.randint(2, 12, n_test)
})

print("Train Shape :", train.shape)
print("Test Shape :", test.shape)

# ==========================================
# ADD SOME MISSING VALUES
# ==========================================

for col in ["LotArea", "GarageArea", "GrLivArea"]:
    train.loc[
        train.sample(frac=0.05, random_state=42).index,
        col
    ] = np.nan

    test.loc[
        test.sample(frac=0.05, random_state=42).index,
        col
    ] = np.nan

# ==========================================
# HANDLE MISSING VALUES
# ==========================================

train.fillna(train.median(numeric_only=True), inplace=True)
test.fillna(test.median(numeric_only=True), inplace=True)

# ==========================================
# FEATURES AND TARGET
# ==========================================

X = train.drop(["Id", "SalePrice"], axis=1)
y = train["SalePrice"]

X_test = test.drop("Id", axis=1)
test_ids = test["Id"]

# ==========================================
# FEATURE SCALING
# ==========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42
)

# ==========================================
# LINEAR REGRESSION
# ==========================================

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_valid)

print("\n----- Linear Regression -----")
print("R2 Score :", round(r2_score(y_valid, lr_pred), 4))
print("RMSE :", round(
    np.sqrt(mean_squared_error(y_valid, lr_pred)), 2
))

# ==========================================
# RIDGE REGRESSION
# ==========================================

ridge = Ridge()

ridge_params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    ridge,
    ridge_params,
    cv=5,
    scoring="r2"
)

ridge_grid.fit(X_train, y_train)

ridge_best = ridge_grid.best_estimator_
ridge_pred = ridge_best.predict(X_valid)

print("\n----- Ridge Regression -----")
print("Best Parameters :", ridge_grid.best_params_)
print("R2 Score :", round(
    r2_score(y_valid, ridge_pred), 4
))

# ==========================================
# LASSO REGRESSION
# ==========================================

lasso = Lasso(max_iter=10000)

lasso_params = {
    "alpha": [0.001, 0.01, 0.1, 1]
}

lasso_grid = GridSearchCV(
    lasso,
    lasso_params,
    cv=5,
    scoring="r2"
)

lasso_grid.fit(X_train, y_train)

lasso_best = lasso_grid.best_estimator_
lasso_pred = lasso_best.predict(X_valid)

print("\n----- Lasso Regression -----")
print("Best Parameters :", lasso_grid.best_params_)
print("R2 Score :", round(
    r2_score(y_valid, lasso_pred), 4
))

# ==========================================
# GRADIENT BOOSTING REGRESSION
# ==========================================

gbr = GradientBoostingRegressor(
    random_state=42
)

gbr_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4]
}

gbr_grid = GridSearchCV(
    gbr,
    gbr_params,
    cv=3,
    scoring="r2"
)

gbr_grid.fit(X_train, y_train)

best_model = gbr_grid.best_estimator_
gbr_pred = best_model.predict(X_valid)

print("\n----- Gradient Boosting -----")
print("Best Parameters :", gbr_grid.best_params_)
print("R2 Score :", round(
    r2_score(y_valid, gbr_pred), 4
))
print("RMSE :", round(
    np.sqrt(mean_squared_error(y_valid, gbr_pred)), 2
))

# ==========================================
# TRAIN BEST MODEL ON FULL DATA
# ==========================================

best_model.fit(X_scaled, y)

# ==========================================
# PREDICT TEST DATA
# ==========================================

test_predictions = best_model.predict(X_test_scaled)

# ==========================================
# CREATE SUBMISSION FILE
# ==========================================

submission = pd.DataFrame({
    "Id": test_ids,
    "PredictedPrice": test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv created successfully!")

print("\nFirst 10 Predictions:")
print(submission.head(10))

Train Shape : (1000, 14)
Test Shape : (300, 13)

----- Linear Regression -----
R2 Score : 0.831
RMSE : 37233.25

----- Ridge Regression -----
Best Parameters : {'alpha': 1}
R2 Score : 0.831

----- Lasso Regression -----
Best Parameters : {'alpha': 1}
R2 Score : 0.831

----- Gradient Boosting -----
Best Parameters : {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
R2 Score : 0.8125
RMSE : 39218.51

submission.csv created successfully!

First 10 Predictions:
     Id  PredictedPrice
0  1001   153314.096856
1  1002   418558.069189
2  1003   282810.275452
3  1004   329974.291747
4  1005   440231.347153
5  1006   325718.718467
6  1007   464395.463829
7  1008   376131.838538
8  1009   232766.351705
9  1010   272330.732020
